# Workshop 00: Machine Learning Foundations & Panel Data Revision

**Course:** Machine Learning II: Forecasting and Dynamic Modeling  
**Institution:** Lazarski University  
**Instructor:** Leonardo Di Caterina  
**Duration:** 120 Minutes (Laboratory Session)  

---

## Executive Overview & Pedagogical Objectives
This diagnostic workshop establishes the mathematical and programmatic bridge between **Machine Learning I** (cross-sectional tabular data with independent, identically distributed observations) and **Machine Learning II** (longitudinal panel data, dynamic time series, and sequential forecasting).

### 120-Minute Laboratory Map
1. **Part 1 (20m):** Synthetic Panel Data Generation — Multi-index architecture ($N$ entities $\times T$ periods), realistic flaws, and dynamic target generation.
2. **Part 2 (25m):** Pandas Fluency & Leak-Free Engineering — Within-entity temporal shifts, rolling windows, and eliminating cross-entity data contamination.
3. **Part 3 (25m):** Manifold Learning & Embeddings — Linear orthogonal projection (PCA) vs non-linear manifold learning (t-SNE & UMAP).
4. **Part 4 (25m):** The Tree Model Spectrum — Decision Tree $\to$ Random Forest $\to$ XGBoost inside leak-free `Pipeline`s with entity-aware `GroupKFold`.
5. **Part 5 (25m):** Residual Diagnostics & Time-Series Bridge — Residual analysis ($e_{i,t} = y_{i,t} - \hat{y}_{i,t}$), diagnostic plotting, error autocorrelation (ACF), and comparative feature importance.

In [ ]:
# Environment & Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import xgboost as xgb

# Visual aesthetics & reproducibility
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Environment initialized successfully. XGBoost version:", xgb.__version__)

## Part 1: Synthetic Panel Data Generation (20 Minutes)

In economic and financial modeling, we rarely observe purely independent cross-sectional records. Instead, we observe $N$ distinct entities (e.g., countries, firms, financial assets) tracked over $T$ continuous time intervals.

### Task 1 Specification
Implement `generate_panel_data(n_entities=10, n_periods=24, n_features=6, random_state=42)`:
1. Generate base features using `sklearn.datasets.make_classification`.
2. Construct a **`pd.MultiIndex`** with hierarchical levels `['entity_id', 'date']` using monthly start dates (`freq='MS'`).
3. Assign a categorical column `'sector'` (e.g., Technology, Healthcare, Finance, Energy) that remains persistent per entity.
4. Construct a continuous target `'target_growth'` that combines:
   - Non-linear combinations of base features ($0.8 x_0 - 0.5 x_1^2 + 1.2 \sin(x_2)$).
   - Entity-specific random intercepts (unobserved individual heterogeneity $\alpha_i$).
   - First-order autoregressive dynamics ($0.35 y_{i,t-1}$).
   - Gaussian innovation noise $\varepsilon_{i,t} \sim \mathcal{N}(0, 0.4^2)$.
5. Retain a binary classification label `'target'` for legacy validation.
6. Introduce realistic data imperfections: randomly inject ~5% missing values (`NaN`) across numeric features.

In [ ]:
def generate_panel_data(n_entities=10, n_periods=24, n_features=6, random_state=42):
    """
    Generates a synthetic panel dataset with MultiIndex ['entity_id', 'date'].
    Includes numeric features, entity sector, continuous dynamic target, and injected NaNs.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 1 Output
df_panel = generate_panel_data(n_entities=8, n_periods=24, n_features=6, random_state=42)
print(f"Panel Shape: {df_panel.shape}")
print(f"MultiIndex Levels: {df_panel.index.names}")
print(f"Missing Values count per column:\n{df_panel.isna().sum()}")
df_panel.head(8)

## Part 2: Pandas Fluency & Leak-Free Feature Engineering (25 Minutes)

When generating dynamic temporal features (lags, differences, rolling averages) on panel data, a naive `.shift(1)` across the entire DataFrame causes **catastrophic inter-entity data contamination**:
The first observation of Entity $B$ would receive the last period's value of Entity $A$!

### Task 2 Specification
Implement `engineer_panel_features(df)`:
1. Compute `feat_0_lag1`: 1-month lag of `feat_0` computed strictly within each entity.
2. Compute `feat_0_roll3_mean`: 3-month rolling mean of `feat_0` computed strictly within each entity (`min_periods=1`).
3. Ensure the first period of every entity strictly contains `NaN` for `feat_0_lag1` (verifying zero cross-entity leakage).

In [ ]:
def engineer_panel_features(df):
    """
    Computes within-entity lags and rolling statistics without cross-entity leakage.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 2 & Verify Zero Cross-Entity Contamination
df_feat = engineer_panel_features(df_panel)
first_periods = df_feat.groupby(level='entity_id').head(1)
assert first_periods['feat_0_lag1'].isna().all(), "Data leakage detected across entity boundaries!"
print("Validation Passed: All entity initial lag periods are NaN (zero inter-entity leakage).")

# Visual inspection for one entity
ent1_data = df_feat.loc['ENT_01']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ent1_data.index, ent1_data['feat_0'], marker='o', label='feat_0 (Raw)', color='#2563eb')
ax.plot(ent1_data.index, ent1_data['feat_0_lag1'], marker='s', linestyle='--', label='feat_0_lag1 (Shift 1)', color='#f59e0b')
ax.plot(ent1_data.index, ent1_data['feat_0_roll3_mean'], marker='^', label='feat_0_roll3_mean (Rolling 3)', color='#10b981')
ax.set_title("Entity 01: Leak-Free Dynamic Feature Engineering")
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: Dimensionality Reduction & Manifold Learning (25 Minutes)

In multivariate panel analysis, high-dimensional feature spaces present redundancy, collinearity, and visualization bottlenecks. We compare three dimensionality reduction paradigms:
1. **PCA (Principal Component Analysis):** Linear orthogonal projection maximizing global variance.
2. **t-SNE (t-Distributed Stochastic Neighbor Embedding):** Probabilistic non-linear technique preserving local neighborhood clusters.
3. **UMAP (Uniform Manifold Approximation and Projection):** Riemannian geometry / topological manifold learning preserving both local clustering and global macro-structure.

### Task 3 Specification
Implement `run_dimensionality_reduction(X, n_components=2, random_state=42)`:
1. Fit `PCA` and compute explained variance ratio.
2. Fit `TSNE` with adaptive perplexity.
3. Fit `umap.UMAP` with `n_neighbors=15`.
4. Return a dictionary containing `'pca_emb'`, `'pca_explained_var'`, `'tsne_emb'`, and `'umap_emb'`.

In [ ]:
def run_dimensionality_reduction(X, n_components=2, random_state=42):
    """
    Applies PCA, t-SNE, and UMAP to high-dimensional feature matrix X.
    Returns 2D embeddings and PCA explained variance ratio.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 3: Visual Diagnostic Comparison
num_cols = [c for c in df_panel.columns if c.startswith('feat_')]
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_clean = imputer.fit_transform(df_panel[num_cols])
X_scaled = scaler.fit_transform(X_clean)

emb_dict = run_dimensionality_reduction(X_scaled, n_components=2, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sectors = df_panel['sector'].values

sns.scatterplot(x=emb_dict['pca_emb'][:, 0], y=emb_dict['pca_emb'][:, 1], hue=sectors, ax=axes[0], palette='viridis')
axes[0].set_title(f"PCA (Expl. Var: {np.sum(emb_dict['pca_explained_var']):.1%})")

sns.scatterplot(x=emb_dict['tsne_emb'][:, 0], y=emb_dict['tsne_emb'][:, 1], hue=sectors, ax=axes[1], palette='viridis')
axes[1].set_title("t-SNE Embedding")

sns.scatterplot(x=emb_dict['umap_emb'][:, 0], y=emb_dict['umap_emb'][:, 1], hue=sectors, ax=axes[2], palette='viridis')
axes[2].set_title("UMAP Topological Projection")

for ax in axes:
    ax.set_xlabel("Dim 1")
    ax.set_ylabel("Dim 2")
plt.tight_layout()
plt.show()

## Part 4: The Tree Model Spectrum (Tree $\to$ Forest $\to$ Boost) (25 Minutes)

We examine the structural evolution of tree-based algorithms on panel tabular features:
1. **Single Decision Tree (`DecisionTreeRegressor`):**
   - Recursive binary splitting. High variance, vulnerable to overfitting.
2. **Random Forest (`RandomForestRegressor`):**
   - Bagging (Bootstrap Aggregation) + random feature subspace sampling. Reduces variance without increasing bias.
3. **XGBoost (`xgb.XGBRegressor`):**
   - Gradient Boosting: Sequential ensemble fitting new trees to the pseudo-residuals of prior trees with regularization ($L_1, L_2$) and shrinkage.

### Cross-Validation Strategy: `GroupKFold` by `entity_id`
If we randomly shuffle panel records, cross-validation would test observations from the *same* entities observed in training, causing cross-sectional data snooping. We strictly enforce **`GroupKFold` partitioned by `entity_id`**.

### Task 4 Specification
Implement:
1. `build_tree_pipeline(model_name, numeric_features, categorical_features, random_state=42)`: Builds a `ColumnTransformer` (median imputer + standard scaler for numerics, one-hot encoder for categoricals) chained with the requested regressor (`'decision_tree'`, `'random_forest'`, or `'xgboost'`).
2. `train_and_evaluate_trees(df, feature_cols, categorical_cols, target_col='target_growth', n_splits=5, random_state=42)`: Evaluates all three models via `GroupKFold` on `entity_id`, recording out-of-fold predictions, RMSE, MAE, and fitted pipelines.

In [ ]:
def build_tree_pipeline(model_name: str, numeric_features: list, categorical_features: list, random_state: int = 42) -> Pipeline:
    """
    Constructs a ColumnTransformer chained with DecisionTree, RandomForest, or XGBoost.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")

def build_panel_pipeline(numeric_features: list, categorical_features: list) -> Pipeline:
    """Backward-compatibility helper: defaults to RandomForest pipeline."""
    return build_tree_pipeline('random_forest', numeric_features, categorical_features)

def train_and_evaluate_trees(df: pd.DataFrame, feature_cols: list, categorical_cols: list, target_col: str = 'target_growth', n_splits: int = 5, random_state: int = 42) -> dict:
    """
    Evaluates DecisionTree, RandomForest, and XGBoost using GroupKFold grouped by entity_id.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 4: Compare Decision Tree vs Random Forest vs XGBoost
tree_results = train_and_evaluate_trees(
    df_panel, 
    feature_cols=num_cols, 
    categorical_cols=['sector'],
    target_col='target_growth',
    n_splits=4
)

metrics_df = pd.DataFrame({
    m: {
        'OOF RMSE': tree_results['models'][m]['rmse'],
        'OOF MAE': tree_results['models'][m]['mae']
    }
    for m in tree_results['models']
}).T

print("=== GroupKFold Cross-Validation Performance ===")
display(metrics_df)

# Bar plot comparison
fig, ax = plt.subplots(figsize=(8, 4))
metrics_df.plot(kind='bar', ax=ax, colormap='viridis')
ax.set_title("Out-of-Fold Error by Model Architecture")
ax.set_ylabel("Error Metric")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## Part 5: Residual Diagnostics & The Time-Series Bridge (25 Minutes)

Residual analysis is the foundation of dynamic time series modeling. A model's errors are defined as:
$$e_{i,t} = y_{i,t} - \hat{y}_{i,t}$$

If a model has captured all systematic relationships, its residuals should behave as **white noise**: zero mean, constant variance, and zero autocorrelation across time lags.

### The Pedagogical Discovery
Because our synthetic data contains autoregressive dynamics ($y_{i,t} \sim 0.35 y_{i,t-1}$), static tree algorithms (which ignore the sequential ordering of observations) leave **statistically significant residual autocorrelation**:
$$\text{Corr}(e_t, e_{t-1}) > 0$$
This empirical result proves why pure cross-sectional tabular machine learning fails on temporal data, and motivates the dynamic forecasting techniques of **Machine Learning II**!

### Task 5 Specification
Implement `analyze_residuals(y_true, y_pred)`:
1. Compute residuals: $e = y - \hat{y}$.
2. Compute `rmse`, `mae`, and $R^2$.
3. Compute `lag1_autocorr`: Pearson correlation between $e_t$ and $e_{t-1}$.

In [ ]:
def analyze_residuals(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Computes residual errors, error statistics (RMSE, MAE, R2), and lag-1 autocorrelation.
    """
    # -------------------------------------------------------------------------
    # YOUR CODE HERE
    # -------------------------------------------------------------------------
    raise NotImplementedError("Student implementation missing")


In [ ]:
# Demonstrate Task 5: 4-Panel Diagnostic Dashboard & Feature Importances
y_true = tree_results['y_true']
xgb_preds = tree_results['models']['xgboost']['oof_predictions']
diag = analyze_residuals(y_true, xgb_preds)
residuals = diag['residuals']

print(f"=== XGBoost Residual Analysis ===")
print(f"RMSE: {diag['rmse']:.4f} | MAE: {diag['mae']:.4f} | R2: {diag['r2']:.4f}")
print(f"Lag-1 Residual Autocorrelation: {diag['lag1_autocorr']:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Actual vs Fitted
axes[0, 0].scatter(y_true, xgb_preds, alpha=0.6, color='#2563eb', edgecolor='none')
p_min, p_max = min(y_true.min(), xgb_preds.min()), max(y_true.max(), xgb_preds.max())
axes[0, 0].plot([p_min, p_max], [p_min, p_max], 'r--', lw=2, label='45° Line')
axes[0, 0].set_title("Actual vs. Predicted Growth")
axes[0, 0].set_xlabel("Actual Growth")
axes[0, 0].set_ylabel("Predicted Growth")
axes[0, 0].legend()

# Panel 2: Residuals vs Fitted
axes[0, 1].scatter(xgb_preds, residuals, alpha=0.6, color='#0891b2', edgecolor='none')
axes[0, 1].axhline(0, color='r', linestyle='--', lw=2)
axes[0, 1].set_title("Residuals vs. Fitted Values")
axes[0, 1].set_xlabel("Fitted Values")
axes[0, 1].set_ylabel("Residual (e = y - y_hat)")

# Panel 3: Error Distribution & Normality
sns.histplot(residuals, kde=True, ax=axes[1, 0], color='#8b5cf6')
axes[1, 0].axvline(0, color='r', linestyle='--')
axes[1, 0].set_title("Residual Error Distribution")
axes[1, 0].set_xlabel("Residual Value")

# Panel 4: Residual Lag-1 Autocorrelation Scatter
axes[1, 1].scatter(residuals[:-1], residuals[1:], alpha=0.6, color='#d97706')
axes[1, 1].axhline(0, color='gray', linestyle=':', lw=1)
axes[1, 1].axvline(0, color='gray', linestyle=':', lw=1)
axes[1, 1].set_title(f"Lag-1 Residual Scatter (r = {diag['lag1_autocorr']:.3f})")
axes[1, 1].set_xlabel("Residual (t-1)")
axes[1, 1].set_ylabel("Residual (t)")

plt.tight_layout()
plt.show()

# Comparative Feature Importance
fitted_dt = tree_results['models']['decision_tree']['pipeline'].named_steps['regressor']
fitted_rf = tree_results['models']['random_forest']['pipeline'].named_steps['regressor']
fitted_xgb = tree_results['models']['xgboost']['pipeline'].named_steps['regressor']
prep = tree_results['models']['xgboost']['pipeline'].named_steps['preprocessor']
feature_names = prep.get_feature_names_out()

imp_df = pd.DataFrame({
    'Decision Tree': fitted_dt.feature_importances_,
    'Random Forest': fitted_rf.feature_importances_,
    'XGBoost': fitted_xgb.feature_importances_
}, index=feature_names)

fig, ax = plt.subplots(figsize=(10, 5))
imp_df.plot(kind='barh', ax=ax)
ax.set_title("Comparative Feature Importances across Tree Models")
ax.set_xlabel("Relative Importance Score")
plt.tight_layout()
plt.show()

## Summary & Bridge to Workshop 01

### What We Learned Today
1. **Panel Data Architecture:** Multi-index structure allows tracking both cross-sectional entities and temporal dynamics simultaneously.
2. **Preventing Temporal Leakage:** Feature shifts and rolling computations must strictly happen within individual entity groups.
3. **Manifold Learning:** PCA provides orthogonal variance baselines, while t-SNE and UMAP uncover non-linear cluster topologies.
4. **The Tree Spectrum:** From high-variance Decision Trees to variance-reduced Random Forests and bias-corrected XGBoost.
5. **Residuals Never Lie:** Static cross-sectional machine learning models leave autocorrelated residuals when applied to dynamic time processes.

### Coming Up in Workshop 01: Time Series Graphics & Diagnostics
In **Workshop 01**, we will pull real macroeconomic and financial time series directly from the Federal Reserve (FRED) API, decompose structural trends and seasonality, and compute formal Autocorrelation Functions (ACF) to model dynamic processes directly.